# Fused Attention CUDA Kernel — Colab

Builds, tests, and benchmarks a fused tiled attention kernel with online streaming softmax.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

**What this does NOT do on Colab:** Nsight Compute (`ncu`) profiling — the free tier blocks GPU performance counter access. Everything else works: compilation, correctness tests, and `cudaEvent` benchmarks.

**Time:** ~5 minutes total.

## 0. Verify GPU Access

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then re-run.'
)
gpu = torch.cuda.get_device_name(0)
print(f'{gpu} | torch {torch.__version__} | CUDA {torch.version.cuda}')

r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, 'nvcc not found'

In [ ]:
# Detect GPU compute capability for the -arch flag
cap = torch.cuda.get_device_capability()
ARCH = f'sm_{cap[0]}{cap[1]}'
print(f'Compute capability: {cap[0]}.{cap[1]} -> {ARCH}')

## 1. Write Source Files

Each cell writes one file to `/content/fused-attention/`.

In [ ]:
import os
os.makedirs('/content/fused-attention', exist_ok=True)
os.chdir('/content/fused-attention')
print(f'Working directory: {os.getcwd()}')

In [ ]:
%%writefile attention.cuh
#pragma once

#include <cuda_runtime.h>
#include <cstdio>
#include <cmath>
#include <cfloat>

#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__,  \
                    cudaGetErrorString(err));                                   \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

#ifndef TILE_Q
#define TILE_Q 16
#endif

#ifndef TILE_KV
#define TILE_KV 16
#endif

#ifndef HEAD_DIM
#define HEAD_DIM 64
#endif

__host__ __device__ inline int cdiv(int a, int b) { return (a + b - 1) / b; }

void naive_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal,
    float* workspace = nullptr);

void fused_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal);

inline float* load_bin(const char* path, size_t num_floats) {
    FILE* f = fopen(path, "rb");
    if (!f) { fprintf(stderr, "Cannot open %s\n", path); exit(1); }
    float* buf = (float*)malloc(num_floats * sizeof(float));
    size_t nread = fread(buf, sizeof(float), num_floats, f);
    if (nread != num_floats) {
        fprintf(stderr, "%s: expected %zu floats, got %zu\n", path, num_floats, nread);
        exit(1);
    }
    fclose(f);
    return buf;
}

// numpy-style allclose: |a-b| <= atol + rtol * |b|
// Pure relative error blows up near zero; this handles it properly.
inline bool check(const float* a, const float* b, size_t n,
                  float rtol, const char* label) {
    const float atol = 1e-5f;
    float max_abs = 0, max_rel = 0;
    size_t worst_abs_i = 0;
    int num_fail = 0;
    for (size_t i = 0; i < n; i++) {
        float diff = fabsf(a[i] - b[i]);
        float tol = atol + rtol * fabsf(b[i]);
        if (diff > max_abs) { max_abs = diff; worst_abs_i = i; }
        float denom = fmaxf(fabsf(b[i]), 1e-8f);
        float rel = diff / denom;
        if (rel > max_rel) max_rel = rel;
        if (diff > tol) num_fail++;
    }
    bool pass = (num_fail == 0);
    printf("  %-25s max_abs=%.2e  max_rel=%.2e  fail=%d/%zu  [%s]\n",
           label, max_abs, max_rel, num_fail, n, pass ? "PASS" : "FAIL");
    if (!pass) {
        printf("    worst at i=%zu: got %.6f expected %.6f (diff=%.2e)\n",
               worst_abs_i, a[worst_abs_i], b[worst_abs_i], max_abs);
    }
    return pass;
}

In [ ]:
%%writefile naive_attention.cu
#include "attention.cuh"

__global__ void naive_compute_scores(
    const float* __restrict__ Q,
    const float* __restrict__ K,
    float* __restrict__ S,
    int N, int d, bool causal)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.z * blockDim.x + threadIdx.x;
    if (row >= N || col >= N) return;

    float scale = rsqrtf((float)d);
    if (causal && col > row) {
        S[bh * N * N + row * N + col] = -INFINITY;
        return;
    }

    const float* q_row = Q + bh * N * d + row * d;
    const float* k_col = K + bh * N * d + col * d;
    float dot = 0.0f;
    for (int i = 0; i < d; i++) dot += q_row[i] * k_col[i];
    S[bh * N * N + row * N + col] = dot * scale;
}

__global__ void naive_softmax_rows(float* __restrict__ S, int N)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.x + threadIdx.x;
    if (row >= N) return;

    float* s_row = S + bh * N * N + row * N;
    float m = -INFINITY;
    for (int j = 0; j < N; j++) m = fmaxf(m, s_row[j]);
    float sum = 0.0f;
    for (int j = 0; j < N; j++) {
        s_row[j] = expf(s_row[j] - m);
        sum += s_row[j];
    }
    float inv_sum = 1.0f / sum;
    for (int j = 0; j < N; j++) s_row[j] *= inv_sum;
}

__global__ void naive_attn_times_v(
    const float* __restrict__ S,
    const float* __restrict__ V,
    float* __restrict__ O,
    int N, int d)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int dim = blockIdx.z * blockDim.x + threadIdx.x;
    if (row >= N || dim >= d) return;

    const float* s_row = S + bh * N * N + row * N;
    float acc = 0.0f;
    for (int j = 0; j < N; j++)
        acc += s_row[j] * V[bh * N * d + j * d + dim];
    O[bh * N * d + row * d + dim] = acc;
}

void naive_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal,
    float* workspace)
{
    int BH = B * H;
    float* S = workspace;
    bool own_S = (S == nullptr);
    if (own_S)
        CUDA_CHECK(cudaMalloc(&S, (size_t)BH * N * N * sizeof(float)));

    {
        dim3 block(16, 16);
        dim3 grid(BH, cdiv(N, 16), cdiv(N, 16));
        naive_compute_scores<<<grid, block>>>(Q, K, S, N, d, causal);
    }
    {
        int threads = 256;
        dim3 grid(BH, cdiv(N, threads));
        naive_softmax_rows<<<grid, threads>>>(S, N);
    }
    {
        dim3 block(16, 16);
        dim3 grid(BH, cdiv(N, 16), cdiv(d, 16));
        naive_attn_times_v<<<grid, block>>>(S, V, O, N, d);
    }

    CUDA_CHECK(cudaGetLastError());
    if (own_S) {
        CUDA_CHECK(cudaDeviceSynchronize());
        CUDA_CHECK(cudaFree(S));
    }
}

In [ ]:
%%writefile fused_attention.cu
#include "attention.cuh"

__global__ void fused_attention_kernel(
    const float* __restrict__ Q,
    const float* __restrict__ K,
    const float* __restrict__ V,
    float* __restrict__ O,
    int N, int d, bool causal)
{
    const int bh = blockIdx.x;
    const int q_start = blockIdx.y * TILE_Q;
    const int ty = threadIdx.y;
    const int tx = threadIdx.x;
    const int q_row = q_start + ty;

    const float scale = rsqrtf((float)d);

    __shared__ float Q_s[TILE_Q][HEAD_DIM];
    __shared__ float K_s[TILE_KV][HEAD_DIM];
    __shared__ float V_s[TILE_KV][HEAD_DIM];
    __shared__ float S_s[TILE_Q][TILE_KV];

    float acc[HEAD_DIM / 32];
    for (int i = 0; i < HEAD_DIM / 32; i++) acc[i] = 0.0f;

    float row_max = -INFINITY;
    float row_sum = 0.0f;

    // Load Q tile into shared memory
    for (int dd = tx; dd < d; dd += 32) {
        if (q_row < N)
            Q_s[ty][dd] = Q[bh * N * d + q_row * d + dd];
        else
            Q_s[ty][dd] = 0.0f;
    }
    __syncthreads();

    // Loop over K/V tiles
    int num_kv_tiles = cdiv(N, TILE_KV);
    for (int kv_tile = 0; kv_tile < num_kv_tiles; kv_tile++) {
        int kv_start = kv_tile * TILE_KV;

        // Causal early exit: entire KV tile is past the diagonal
        if (causal && kv_start > q_start + TILE_Q - 1) break;

        // Load K tile
        for (int r = ty; r < TILE_KV; r += TILE_Q) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32) {
                if (kv_row < N)
                    K_s[r][dd] = K[bh * N * d + kv_row * d + dd];
                else
                    K_s[r][dd] = 0.0f;
            }
        }

        // Load V tile
        for (int r = ty; r < TILE_KV; r += TILE_Q) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32) {
                if (kv_row < N)
                    V_s[r][dd] = V[bh * N * d + kv_row * d + dd];
                else
                    V_s[r][dd] = 0.0f;
            }
        }
        __syncthreads();

        // Compute S = Q_s @ K_s^T * scale with warp reduction
        for (int j = 0; j < TILE_KV; j++) {
            int kv_col = kv_start + j;
            float dot = 0.0f;
            for (int dd = tx; dd < d; dd += 32)
                dot += Q_s[ty][dd] * K_s[j][dd];

            for (int offset = 16; offset > 0; offset >>= 1)
                dot += __shfl_down_sync(0xffffffff, dot, offset);

            if (tx == 0) {
                float s = dot * scale;
                if (causal && kv_col > q_row) s = -INFINITY;
                if (q_row >= N || kv_col >= N) s = -INFINITY;
                S_s[ty][j] = s;
            }
        }
        __syncthreads();

        // Online softmax: find tile max, compute correction
        float tile_max = -INFINITY;
        if (tx == 0) {
            for (int j = 0; j < TILE_KV; j++)
                tile_max = fmaxf(tile_max, S_s[ty][j]);
        }
        tile_max = __shfl_sync(0xffffffff, tile_max, 0);

        float m_new = fmaxf(row_max, tile_max);
        float correction = expf(row_max - m_new);

        float tile_sum = 0.0f;
        if (tx == 0) {
            for (int j = 0; j < TILE_KV; j++) {
                S_s[ty][j] = expf(S_s[ty][j] - m_new);
                tile_sum += S_s[ty][j];
            }
        }
        tile_sum = __shfl_sync(0xffffffff, tile_sum, 0);
        __syncthreads();

        row_sum = row_sum * correction + tile_sum;

        // Rescale accumulator and add P @ V_tile
        for (int i = 0; i < HEAD_DIM / 32; i++) {
            int dd = tx + i * 32;
            acc[i] *= correction;
            for (int j = 0; j < TILE_KV; j++)
                acc[i] += S_s[ty][j] * V_s[j][dd];
        }

        row_max = m_new;
        __syncthreads();
    }

    // Write output: O = acc / row_sum
    if (q_row < N) {
        float inv_sum = (row_sum > 0.0f) ? (1.0f / row_sum) : 0.0f;
        for (int i = 0; i < HEAD_DIM / 32; i++) {
            int dd = tx + i * 32;
            if (dd < d)
                O[bh * N * d + q_row * d + dd] = acc[i] * inv_sum;
        }
    }
}

void fused_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal)
{
    int BH = B * H;
    dim3 block(32, TILE_Q);
    dim3 grid(BH, cdiv(N, TILE_Q));
    fused_attention_kernel<<<grid, block>>>(Q, K, V, O, N, d, causal);
    CUDA_CHECK(cudaGetLastError());
}

In [ ]:
%%writefile main.cu
#include "attention.cuh"
#include <cstdlib>
#include <cstring>
#include <cstdio>

int run_test(const char* data_dir) {
    char path[512];
    snprintf(path, sizeof(path), "%s/meta.json", data_dir);
    FILE* mf = fopen(path, "r");
    if (!mf) { fprintf(stderr, "Cannot open %s\n", path); return 1; }
    char meta_buf[1024];
    size_t meta_len = fread(meta_buf, 1, sizeof(meta_buf) - 1, mf);
    (void)meta_len;
    meta_buf[sizeof(meta_buf) - 1] = 0;
    fclose(mf);

    auto parse_int = [&](const char* key) -> int {
        const char* p = strstr(meta_buf, key);
        if (!p) { fprintf(stderr, "Missing key: %s\n", key); exit(1); }
        p = strchr(p, ':');
        return atoi(p + 1);
    };

    int B = parse_int("\"B\"");
    int H = parse_int("\"H\"");
    int N = parse_int("\"N\"");
    int d = parse_int("\"d\"");
    bool causal = strstr(meta_buf, "\"causal\": true") != nullptr;

    printf("Test: B=%d H=%d N=%d d=%d causal=%d  [%s]\n", B, H, N, d, causal, data_dir);

    size_t qkv_size = (size_t)B * H * N * d;
    snprintf(path, sizeof(path), "%s/Q.bin", data_dir);
    float* h_Q = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/K.bin", data_dir);
    float* h_K = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/V.bin", data_dir);
    float* h_V = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/O_ref.bin", data_dir);
    float* h_O_ref = load_bin(path, qkv_size);

    float *d_Q, *d_K, *d_V, *d_O;
    size_t bytes = qkv_size * sizeof(float);
    CUDA_CHECK(cudaMalloc(&d_Q, bytes));
    CUDA_CHECK(cudaMalloc(&d_K, bytes));
    CUDA_CHECK(cudaMalloc(&d_V, bytes));
    CUDA_CHECK(cudaMalloc(&d_O, bytes));
    CUDA_CHECK(cudaMemcpy(d_Q, h_Q, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_K, h_K, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, bytes, cudaMemcpyHostToDevice));

    float* h_O_out = (float*)malloc(bytes);
    bool all_pass = true;

    CUDA_CHECK(cudaMemset(d_O, 0, bytes));
    naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
    CUDA_CHECK(cudaMemcpy(h_O_out, d_O, bytes, cudaMemcpyDeviceToHost));
    all_pass &= check(h_O_out, h_O_ref, qkv_size, 1e-4f, "naive vs reference");

    CUDA_CHECK(cudaMemset(d_O, 0, bytes));
    fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_O_out, d_O, bytes, cudaMemcpyDeviceToHost));
    all_pass &= check(h_O_out, h_O_ref, qkv_size, 1e-4f, "fused vs reference");

    free(h_Q); free(h_K); free(h_V); free(h_O_ref); free(h_O_out);
    CUDA_CHECK(cudaFree(d_Q));
    CUDA_CHECK(cudaFree(d_K));
    CUDA_CHECK(cudaFree(d_V));
    CUDA_CHECK(cudaFree(d_O));
    return all_pass ? 0 : 1;
}

void run_bench() {
    int B = 1, H = 8, d = 64;
    bool causal = true;
    int Ns[] = {128, 256, 512, 1024, 2048};
    int num_sizes = sizeof(Ns) / sizeof(Ns[0]);
    int warmup = 10, iters = 100;

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s (SM %d.%d, %d SMs)\n",
           prop.name, prop.major, prop.minor, prop.multiProcessorCount);
    printf("B=%d H=%d d=%d causal=%d  warmup=%d iters=%d\n\n",
           B, H, d, causal, warmup, iters);
    printf("%-8s  %12s  %12s  %8s\n", "N", "Naive (ms)", "Fused (ms)", "Speedup");
    printf("----------------------------------------------\n");

    for (int ni = 0; ni < num_sizes; ni++) {
        int N = Ns[ni];
        size_t qkv_size = (size_t)B * H * N * d;
        size_t bytes = qkv_size * sizeof(float);

        float *d_Q, *d_K, *d_V, *d_O;
        CUDA_CHECK(cudaMalloc(&d_Q, bytes));
        CUDA_CHECK(cudaMalloc(&d_K, bytes));
        CUDA_CHECK(cudaMalloc(&d_V, bytes));
        CUDA_CHECK(cudaMalloc(&d_O, bytes));

        float* h_tmp = (float*)malloc(bytes);
        srand(42);
        for (size_t i = 0; i < qkv_size; i++)
            h_tmp[i] = ((float)rand() / RAND_MAX - 0.5f) * 2.0f;
        CUDA_CHECK(cudaMemcpy(d_Q, h_tmp, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_K, h_tmp, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_V, h_tmp, bytes, cudaMemcpyHostToDevice));
        free(h_tmp);

        float* naive_ws;
        CUDA_CHECK(cudaMalloc(&naive_ws, (size_t)B * H * N * N * sizeof(float)));

        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        for (int i = 0; i < warmup; i++)
            naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal, naive_ws);
        CUDA_CHECK(cudaEventRecord(start));
        for (int i = 0; i < iters; i++)
            naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal, naive_ws);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float naive_ms;
        CUDA_CHECK(cudaEventElapsedTime(&naive_ms, start, stop));
        naive_ms /= iters;

        for (int i = 0; i < warmup; i++)
            fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
        CUDA_CHECK(cudaEventRecord(start));
        for (int i = 0; i < iters; i++)
            fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float fused_ms;
        CUDA_CHECK(cudaEventElapsedTime(&fused_ms, start, stop));
        fused_ms /= iters;

        printf("%-8d  %12.4f  %12.4f  %7.2fx\n",
               N, naive_ms, fused_ms, naive_ms / fused_ms);

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
        CUDA_CHECK(cudaFree(naive_ws));
        CUDA_CHECK(cudaFree(d_Q));
        CUDA_CHECK(cudaFree(d_K));
        CUDA_CHECK(cudaFree(d_V));
        CUDA_CHECK(cudaFree(d_O));
    }
}

int main(int argc, char** argv) {
    if (argc < 2) {
        printf("Usage: %s test [dir] | bench\n", argv[0]);
        return 1;
    }
    if (strcmp(argv[1], "test") == 0) {
        const char* dir = (argc > 2) ? argv[2] : "test_data";
        int rc = run_test(dir);
        if (rc == 0) {
            int extra_ns[] = {37, 127, 200};
            for (int i = 0; i < 3; i++) {
                char subdir[256];
                snprintf(subdir, sizeof(subdir), "%s/N%d", dir, extra_ns[i]);
                char meta_path[300];
                snprintf(meta_path, sizeof(meta_path), "%s/meta.json", subdir);
                FILE* f = fopen(meta_path, "r");
                if (f) { fclose(f); rc |= run_test(subdir); }
            }
        }
        printf("\n%s\n", rc == 0 ? "ALL TESTS PASSED" : "SOME TESTS FAILED");
        return rc;
    } else if (strcmp(argv[1], "bench") == 0) {
        run_bench();
        return 0;
    }
    fprintf(stderr, "Unknown command: %s\n", argv[1]);
    return 1;
}

In [ ]:
%%writefile generate_reference.py
import argparse, json, os
import torch
import torch.nn.functional as F

def naive_attention(Q, K, V, causal=True):
    B, H, N, d = Q.shape
    scale = d ** -0.5
    S = (Q @ K.transpose(-2, -1)) * scale
    if causal:
        mask = torch.triu(torch.ones(N, N, device=Q.device, dtype=torch.bool), diagonal=1)
        S = S.masked_fill(mask, float('-inf'))
    return torch.softmax(S, dim=-1) @ V

def save_tensor(t, path):
    t = t.contiguous().float()
    with open(path, 'wb') as f:
        f.write(t.numpy().tobytes())

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--B', type=int, default=1)
    p.add_argument('--H', type=int, default=1)
    p.add_argument('--N', type=int, default=128)
    p.add_argument('--d', type=int, default=64)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--causal', action='store_true', default=True)
    p.add_argument('--outdir', type=str, default='test_data')
    args = p.parse_args()

    torch.manual_seed(args.seed)
    Q = torch.randn(args.B, args.H, args.N, args.d)
    K = torch.randn(args.B, args.H, args.N, args.d)
    V = torch.randn(args.B, args.H, args.N, args.d)
    O = naive_attention(Q, K, V, causal=args.causal)

    os.makedirs(args.outdir, exist_ok=True)
    save_tensor(Q, f'{args.outdir}/Q.bin')
    save_tensor(K, f'{args.outdir}/K.bin')
    save_tensor(V, f'{args.outdir}/V.bin')
    save_tensor(O, f'{args.outdir}/O_ref.bin')

    meta = {'B': args.B, 'H': args.H, 'N': args.N, 'd': args.d,
            'causal': args.causal, 'seed': args.seed, 'dtype': 'float32'}
    with open(f'{args.outdir}/meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'Saved ({args.B},{args.H},{args.N},{args.d}) to {args.outdir}/')

    for test_N in [37, 127, 200]:
        sub = f'{args.outdir}/N{test_N}'
        os.makedirs(sub, exist_ok=True)
        torch.manual_seed(args.seed)
        Qq = torch.randn(args.B, args.H, test_N, args.d)
        Kk = torch.randn(args.B, args.H, test_N, args.d)
        Vv = torch.randn(args.B, args.H, test_N, args.d)
        Oo = naive_attention(Qq, Kk, Vv, causal=args.causal)
        save_tensor(Qq, f'{sub}/Q.bin')
        save_tensor(Kk, f'{sub}/K.bin')
        save_tensor(Vv, f'{sub}/V.bin')
        save_tensor(Oo, f'{sub}/O_ref.bin')
        with open(f'{sub}/meta.json', 'w') as f:
            json.dump({**meta, 'N': test_N}, f, indent=2)
        print(f'  + N={test_N}')

if __name__ == '__main__':
    main()

## 2. Generate PyTorch Reference Data

In [ ]:
!python generate_reference.py

## 3. Build CUDA Kernels

In [ ]:
!nvcc -O3 -std=c++17 --use_fast_math -arch={ARCH} \
    -DTILE_Q=16 -DTILE_KV=16 -DHEAD_DIM=64 \
    -o attention main.cu naive_attention.cu fused_attention.cu
print('Build successful')

## 4. Correctness Tests

Both kernels checked against PyTorch reference using numpy-style `allclose`: `|a-b| <= 1e-5 + 1e-4 * |b|`. Includes non-power-of-two sequence lengths (N=37, 127, 200).

In [ ]:
!./attention test test_data

## 5. Benchmark

Sweeps N from 128 to 2048. Timing uses `cudaEvent` (not host timers). 10 warmup iterations discarded, 100 timed.

The speedup grows with N — that's the whole point of eliminating the N×N materialization.

In [ ]:
!./attention bench

## 6. Compare Against PyTorch SDPA

This is NOT the baseline for the resume bullet (that's the naive CUDA kernel above). This is the honesty check — how far are we from a production kernel?

In [ ]:
import torch
import torch.nn.functional as F

device = 'cuda'
B, H, d = 1, 8, 64
causal = True
warmup, iters = 10, 100

print(f'PyTorch SDPA benchmark (B={B}, H={H}, d={d}, causal={causal})')
print(f'{"N":<8}  {"SDPA (ms)":>12}')
print('-' * 24)

for N in [128, 256, 512, 1024, 2048]:
    Q = torch.randn(B, H, N, d, device=device)
    K = torch.randn(B, H, N, d, device=device)
    V = torch.randn(B, H, N, d, device=device)

    for _ in range(warmup):
        F.scaled_dot_product_attention(Q, K, V, is_causal=causal)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        F.scaled_dot_product_attention(Q, K, V, is_causal=causal)
    end.record()
    torch.cuda.synchronize()

    ms = start.elapsed_time(end) / iters
    print(f'{N:<8}  {ms:>12.4f}')

print('\nCompare these against the fused kernel times above.')
print('Reaching ~30-50% of SDPA is expected for a hand-rolled first kernel.')

## 7. Download Results

Grab the benchmark output to fill into your README.

In [ ]:
!./attention bench > benchmark_results.txt 2>&1
!cat benchmark_results.txt

from google.colab import files
files.download('benchmark_results.txt')

## Notes

**What you have after running this notebook:**
- Correctness verified against PyTorch reference (fp32, allclose atol=1e-5 rtol=1e-4)
- Speedup numbers: fused vs naive CUDA baseline at N=128..2048
- SDPA comparison for the honesty section

**What you don't have (Colab limitation):**
- Nsight Compute profiling (achieved occupancy, memory throughput, warp stalls)
- To get those, use an AWS/Lambda/vast.ai instance with root access

**Resume bullet without ncu:**
> Wrote a fused attention CUDA kernel with shared-memory tiling and online streaming
> softmax, eliminating N×N score-matrix materialization: [X]x faster than a naive
> CUDA baseline at N=2048 on a T4, verified against a PyTorch reference to 1e-4.